In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

# 從 Hugging Face 載入模型
config = PeftConfig.from_pretrained("coconut19/llama-dialog-lora")
base_model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path)
model = PeftModel.from_pretrained(base_model, "coconut19/llama-dialog-lora")
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

In [3]:
### type your query here ###
query = "Generate a podcast content between two people discussing their favorite class of computer science."

In [4]:
from retrieval import retrieve

results = retrieve(query)

In [ ]:
import torch
model.eval()

def generate_dialog(user_instruction, retrieved_context, max_new_tokens=256):
    prompt = f"""
    Use the information provided in the relevant context to generate the answer.
    ### Relevant Context:
    {retrieved_context}
    ### Instruction:Instruction: {user_instruction}\n ### Answer:\n"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text[len(prompt):].strip()

retrieved_texts = results[0]['text']
llm_output = generate_dialog(query, retrieved_texts)

In [5]:
import re

def parse_dialog_to_kokoro(dialog_text):
    lines = dialog_text.strip().split('\n')
    result = []
    for line in lines:
        m = re.match(r'^([AB]):\s*(.*)', line.strip())
        if m:
            speaker, text = m.groups()
            result.append((speaker, text))
    return result

kokoro_dialog = parse_dialog_to_kokoro(llm_output)
print(kokoro_dialog)

[('A', "You know, I'm really curious about what kind of programming languages are used by computer science students."), ('B', 'Well, most students learn in C, and C is the most widely used programming language today.'), ('A', "That's right! I've heard that C++ is the most widely used programming language among students today."), ('B', "Actually, it's more accurate to say that most students learn in C, but they use C++ when they have to."), ('A', "I see. I've heard that C++ is a more powerful language than C."), ('B', "Yes, C++ is a more powerful language than C, but it's also more difficult to learn."), ('A', "I've heard that C++ is used in most operating systems, but C is used in most embedded systems."), ('B', "That's right! C is widely used in embedded systems, but C++ is used in most operating systems."), ('A', "I see. Well, I think I've got a good understanding of the subject now."), ('B', "That's great! I'm glad I could help you understand it."), ('A', 'Thanks for the help!')]


In [4]:
!pip install -q kokoro>=0.9.2 soundfile
!apt-get -qq -y install espeak-ng > /dev/null 2>&1
from kokoro import KPipeline
from IPython.display import display, Audio
import soundfile as sf
import torch

In [6]:
pipeline = KPipeline(lang_code='a')

VOICE_MAP = {
    "A": "af_heart",
    "B": "am_adam",
}

for i, (speaker, text) in enumerate(kokoro_dialog):
    print(f"[{i}] {speaker}: {text}")

    generator = pipeline(text, voice=VOICE_MAP[speaker])

    for _, _, audio in generator:
        display(Audio(data=audio, rate=24000, autoplay=(i == 0)))

        outfile = f"{i:02d}_{speaker}.wav"
        sf.write(outfile, audio, 24000)
        print(f"Saved: {outfile}")

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


kokoro-v1_0.pth:   0%|          | 0.00/327M [00:00<?, ?B/s]

[0] A: You know, I'm really curious about what kind of programming languages are used by computer science students.


voices/af_heart.pt:   0%|          | 0.00/523k [00:00<?, ?B/s]

...

Saved: 06_A.wav
[7] B: That's right! C is widely used in embedded systems, but C++ is used in most operating systems.


Saved: 07_B.wav
[8] A: I see. Well, I think I've got a good understanding of the subject now.


Saved: 08_A.wav
[9] B: That's great! I'm glad I could help you understand it.


Saved: 09_B.wav
[10] A: Thanks for the help!


Saved: 10_A.wav


In [9]:
from pydub import AudioSegment
import os

def merge_wavs(wav_files, output="final.wav"):
    audio = AudioSegment.empty()

    for wav in wav_files:
        print("Adding:", wav)
        segment = AudioSegment.from_wav(wav)
        audio += segment

    audio.export(output, format="wav")
    print("Saved merged file:", output)

wav_files = sorted([f for f in os.listdir(".") if f.endswith(".wav")])
merge_wavs(wav_files, "podcast_dialogue.wav")

Adding: 00_A.wav
Adding: 01_B.wav
Adding: 02_A.wav
Adding: 03_B.wav
Adding: 04_A.wav
Adding: 05_B.wav
Adding: 06_A.wav
Adding: 07_B.wav
Adding: 08_A.wav
Adding: 09_B.wav
Adding: 10_A.wav
Saved merged file: podcast_dialogue.wav
